# LeetCode 294 — Flip Game II bằng Python

Link đề: https://leetcode.com/problems/flip-game-ii/ · Độ khó: **Medium** (Premium) · Chủ đề: Backtracking, Memoization, Game Theory (minimax), Sprague–Grundy

Skeleton LeetCode:

```python
class Solution:
    def canWin(self, currentState: str) -> bool:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (minimax thuần không nhớ trên chuỗi ngắn).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trò chơi hai người luân phiên, mỗi lượt chọn **hai dấu `'+'` liền kề** và lật thành `"--"`; **ai không đi được nữa thì thua**. Hỏi người đi **trước** có **đảm bảo thắng** (với cách chơi tối ưu của cả hai) không.
2. Nắm khái niệm **trạng thái thắng/thua (winning/losing position)** của trò chơi tổ hợp **công bằng**: trạng thái là **thắng** nếu **tồn tại** một nước đi dẫn tới trạng thái **thua** của đối thủ; là **thua** nếu **mọi** nước đi dẫn tới trạng thái thắng của đối thủ (hoặc không có nước đi).
3. Đi từ **minimax đệ quy thuần** (hàm mũ) → **thêm bộ nhớ đệm** (memoization) trên chuỗi → **trạng thái dạng mặt nạ bit** → **trạng thái chuẩn hóa theo đa tập độ dài các đoạn `+`** → **lý thuyết Sprague–Grundy** (số Grundy từng đoạn, XOR).
4. Hiểu vì sao các đoạn `'+'` **độc lập** (một nước đi chỉ ảnh hưởng đúng một đoạn và có thể **tách nó thành hai đoạn**) nên trò chơi là **tổng của các trò chơi con** — nền của Sprague–Grundy.
5. Tránh các lỗi kinh điển: quên điều kiện **không còn nước đi ⇒ thua**, nhầm vai người chơi khi đệ quy, **không nhớ** các trạng thái đã tính, kết luận bằng **tính chẵn lẻ** số cặp `"++"`, và lật sai vị trí.

## 2. Đề bài & ví dụ

Bạn đang chơi **Flip Game** (trò lật) với bạn của mình. Cho chuỗi `currentState` chỉ gồm `'+'` và `'-'`. Hai người **thay phiên** nhau lật **hai ký tự `"++"` liên tiếp** thành `"--"`. Trò chơi **kết thúc** khi một người **không thể đi nữa**; khi đó người còn lại là **người thắng**.

Trả về `true` nếu **người đi trước có thể đảm bảo thắng**, ngược lại trả `false`.

**Ví dụ 1**

- Input: `currentState = "++++"`
- Output: `true`
- Giải thích: người đi trước thắng chắc bằng cách lật `"++"` ở **giữa** thành `"+--+"` (đối thủ không còn nước đi nào).

**Ví dụ 2**

- Input: `currentState = "+"`
- Output: `false`

**Ràng buộc:** `1 <= currentState.length <= 60`; `currentState[i]` là `'+'` hoặc `'-'`; **không có quá `20` dấu `'+'` liên tiếp**.

**Follow up:** suy ra **độ phức tạp thời gian** của thuật toán của bạn.

In [1]:
from typing import List, Dict, Tuple
from functools import lru_cache
import random
import time

examples = [
    ("++++", True),
    ("+", False),
]
for state, expected in examples:
    print(f"currentState={state!r} -> kỳ vọng {expected}")

currentState='++++' -> kỳ vọng True
currentState='+' -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Nước đi hợp lệ và chuỗi bất biến

Nước đi: chọn chỉ số `i` với `state[i] == state[i+1] == '+'` rồi thay thành `'-'`, `'-'`. Chuỗi Python **bất biến** nên ta **tạo chuỗi mới** bằng cắt lát: `state[:i] + "--" + state[i+2:]`.
#### Code demo

In [2]:
def moves(state):
    """Mọi trạng thái đạt được sau đúng một nước đi."""
    return [state[:i] + "--" + state[i + 2:] for i in range(len(state) - 1) if state[i] == state[i + 1] == "+"]


print(moves("++++"))      # ['--++', '+--+', '++--']
print(moves("+-+"))       # []: không còn nước đi

['--++', '+--+', '++--']
[]


### 3.2. Trạng thái thắng / thua và minimax

Với trò chơi hai người, **công bằng** (mỗi người có cùng tập nước đi), ai không đi được thì thua:

- trạng thái **thắng** (cho người sắp đi): tồn tại nước đi tới trạng thái **thua** (cho đối thủ);
- trạng thái **thua**: **mọi** nước đi tới trạng thái thắng (cho đối thủ), kể cả khi **không có nước đi** nào (thua ngay).

`can_win(state) = any(not can_win(next) for next in moves(state))`. Không có nước đi ⇒ `any([])` là `False` ⇒ thua.
#### Code demo

In [3]:
def can_win_demo(state):
    return any(not can_win_demo(nxt) for nxt in moves(state))


for s in ["+", "++", "+++", "++++", "+-+", "-"]:
    print(f"{s!r}: người đi trước {'thắng' if can_win_demo(s) else 'thua'}")

'+': người đi trước thua
'++': người đi trước thắng
'+++': người đi trước thắng
'++++': người đi trước thắng
'+-+': người đi trước thua
'-': người đi trước thua


### 3.3. Ghi nhớ (memoization)

Nhiều dãy nước đi khác nhau cho **cùng một trạng thái** (ví dụ lật cặp thứ nhất rồi cặp thứ ba, hay ngược lại). Lưu kết quả mỗi trạng thái vào **từ điển** hoặc dùng `functools.lru_cache` để **mỗi trạng thái chỉ tính một lần**.
#### Code demo

In [4]:
calls = {"plain": 0, "memo": 0}


def plain(state):
    calls["plain"] += 1
    return any(not plain(nxt) for nxt in moves(state))


@lru_cache(maxsize=None)
def memoized(state):
    calls["memo"] += 1
    return any(not memoized(nxt) for nxt in moves(state))


s = "+" * 12
print(plain(s) == memoized(s), calls)       # số lời gọi khác biệt rất lớn

True {'plain': 202, 'memo': 94}


### 3.4. Các đoạn `'+'` độc lập: tổng của các trò chơi con

Dấu `'-'` **chặn** các nước đi qua nó: mỗi đoạn `'+'` liên tiếp là một "trò chơi con" riêng. Lật `"++"` trong một đoạn dài `k` (ở vị trí bất kỳ) **tách** nó thành hai đoạn dài `a` và `b` với `a + b = k - 2` (có thể bằng `0`). Mọi nước đi chỉ ảnh hưởng **một** đoạn. Vì vậy trạng thái đầy đủ chỉ cần **đa tập các độ dài đoạn**, không cần vị trí.
#### Code demo

In [5]:
import re


def segments(state):
    return sorted(len(m) for m in re.findall(r"\++", state))


print(segments("++-+++--+"), segments("+-+-+"), segments("----"))   # [1, 2, 3] [1, 1, 1] []

[1, 2, 3] [1, 1, 1] []


### 3.5. Số Grundy (Sprague–Grundy) và `mex`

Với trò chơi tổng của nhiều trò chơi con độc lập, mỗi trò con có một **số Grundy** `g`: `g(trạng thái) = mex{ g(trạng thái sau một nước đi) }`, trong đó `mex(S)` là số nguyên không âm **nhỏ nhất không thuộc `S`**. **Tổng** có số Grundy bằng **XOR** các số Grundy con. Người đi trước **thắng** ⟺ số Grundy tổng `≠ 0`. Ở bài này đoạn dài `k` cho các trạng thái sau một nước đi là **tổng của hai đoạn `a` và `b`** (số Grundy `g[a] ^ g[b]`).
#### Code demo

In [6]:
def mex(values):
    m = 0
    s = set(values)
    while m in s:
        m += 1
    return m


print(mex([0, 1, 3]), mex([]), mex([1, 2]))         # 2 0 0
print(2 ^ 3, 5 ^ 5, 0 ^ 7)                          # 1 0 7

2 0 0
1 0 7


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1 `"++++"`: các nước đi: lật cặp `(0,1)` ⇒ `"--++"`, lật `(1,2)` ⇒ `"+--+"`, lật `(2,3)` ⇒ `"++--"`.

- `"+--+"`: không còn cặp `"++"` ⇒ đối thủ **không đi được ⇒ thua**. Vậy `"++++"` là trạng thái **thắng** (đi vào giữa).
- (Các nước kia dẫn tới `"--++"` hoặc `"++--"`, còn một cặp nên đối thủ đi xong và ta thua — không nên chọn.)

Ví dụ `"+++"`: hai nước đi `"--+"` hoặc `"+--"`, đều khiến đối thủ hết nước ⇒ **thắng**. Ví dụ `"++"`: một nước đi, đối thủ hết nước ⇒ thắng. Còn `"+"` không có nước đi ⇒ **thua**.

**Cách nghĩ:** nhìn từ **kết thúc ngược về**: trạng thái không có nước đi là thua; trạng thái có nước đi tới "thua" là thắng; trạng thái mà mọi nước đi đều tới "thắng" là thua.

## 5. Cách 1 — Brute Force / baseline: minimax đệ quy thuần

**Ý tưởng.** `can_win(state) = any(not can_win(next) for next in moves(state))`, không nhớ gì.
#### Code demo

In [7]:
def can_win_minimax(state):
    for i in range(len(state) - 1):
        if state[i] == "+" and state[i + 1] == "+":
            nxt = state[:i] + "--" + state[i + 2:]
            if not can_win_minimax(nxt):                  # có nước đi làm đối thủ thua -> ta thắng
                return True
    return False                                          # mọi nước đi đều dẫn tới thắng cho đối thủ (hoặc không có nước đi)


for state, expected in examples:
    print(repr(state), "->", can_win_minimax(state), "(kỳ vọng", expected, ")")

'++++' -> True (kỳ vọng True )
'+' -> False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Số nút của cây trò chơi tăng **rất nhanh**: với một đoạn `n` dấu `+`, số nước đi tối đa `~ n - 1` mỗi lượt và độ sâu `~ n/2` ⇒ cỡ `(n - 1)!!` — **bùng nổ giai thừa**; chỉ chạy được tới `n ≈ 14` dấu `+`. **Space `O(n²)`** (độ sâu `n/2` × chuỗi dài `n`).

**Nút thắt.** Cùng một trạng thái được tính lại rất nhiều lần (qua các thứ tự nước đi khác nhau). Chỉ cần tính **một lần** mỗi trạng thái.

## 6. Ý tưởng thuật toán / bước đột phá

**Hướng 1 — Ghi nhớ trạng thái.** Số trạng thái **khác nhau** nhỏ hơn rất nhiều số đường đi; lưu `memo[state]` rồi dùng lại. Thành `O(số trạng thái × n)`.

**Hướng 2 — Chuẩn hóa trạng thái.** Vì các đoạn độc lập, ta chỉ cần **đa tập độ dài các đoạn** (bỏ vị trí): ví dụ `"++-++"` và `"--++++"`... có cùng nhiều ý nghĩa với `(2,2)` / `(4)`; số trạng thái giảm nhiều.

**Hướng 3 — Sprague–Grundy.** Mỗi đoạn dài `k` có số Grundy `g[k] = mex{ g[a] ^ g[b] : a + b = k - 2 }` (tính một lần bằng quy hoạch động `O(n²)`); trạng thái đầy đủ thắng ⟺ XOR các `g[len]` `≠ 0`. **Không cần duyệt cây trò chơi.**

## 7. Các cách tiếp theo

### Cách 2 — Minimax có nhớ trên chuỗi

**Giải thích.** Thêm từ điển `memo` ánh xạ chuỗi trạng thái → kết quả (`lru_cache`).
#### Code demo

In [8]:
def can_win_memo(state):
    memo: Dict[str, bool] = {}

    def win(s):
        if s in memo:
            return memo[s]
        result = False
        for i in range(len(s) - 1):
            if s[i] == "+" and s[i + 1] == "+":
                if not win(s[:i] + "--" + s[i + 2:]):
                    result = True
                    break
        memo[s] = result
        return result

    return win(state)


for state, expected in examples:
    print(repr(state), "->", can_win_memo(state), "(kỳ vọng", expected, ")")

'++++' -> True (kỳ vọng True )
'+' -> False (kỳ vọng False )


In [9]:
# Code nộp LeetCode cho Cách 2 (và Cách 3 nhanh hơn một chút bằng mặt nạ bit)
class Solution:
    def canWin(self, currentState: str) -> bool:
        @lru_cache(None)
        def dfs(mask: int) -> bool:
            for i in range(n - 1):
                if (mask & (1 << i)) == 0 or (mask & (1 << (i + 1))) == 0:
                    continue
                if dfs(mask ^ (1 << i) ^ (1 << (i + 1))):
                    continue
                return True
            return False

        mask, n = 0, len(currentState)
        for i, c in enumerate(currentState):
            if c == "+":
                mask |= 1 << i
        return dfs(mask)


print(Solution().canWin("++++"), Solution().canWin("+"))

True False


**Độ phức tạp Cách 2:** gọi `S` là số trạng thái khác nhau đạt được (tối đa số cách chọn tập các cặp rời nhau để lật). Mỗi trạng thái xét `O(n)` nước đi, mỗi nước tạo chuỗi `O(n)`: **Time `O(S · n²)`**; **Space `O(S · n)`**. `S` tăng cấp số nhân theo độ dài (dãy kiểu Fibonacci cho một đoạn), nhưng **tìm thấy nước thắng thì dừng sớm** nên thực tế nhanh hơn.

### Cách 3 — Minimax có nhớ với trạng thái là mặt nạ bit

**Giải thích.** Mỗi vị trí là một bit: `1` = `'+'`, `0` = `'-'`. Nước đi tại `i` cần bit `i` và `i+1` đều `1`, và **lật cả hai về `0`** bằng `mask ^ (1 << i) ^ (1 << (i + 1))`. Số nguyên dùng làm khóa băm nhanh hơn chuỗi.
#### Code demo

In [10]:
def can_win_bitmask(state):
    n = len(state)

    @lru_cache(maxsize=None)
    def dfs(mask):
        for i in range(n - 1):
            if mask >> i & 1 and mask >> (i + 1) & 1:                     # hai bit liền kề đều là '+'
                if not dfs(mask ^ (1 << i) ^ (1 << (i + 1))):             # lật cả hai về '-'
                    return True
        return False

    mask = 0
    for i, ch in enumerate(state):
        if ch == "+":
            mask |= 1 << i
    result = dfs(mask)
    dfs.cache_clear()
    return result


for state, expected in examples:
    print(repr(state), "->", can_win_bitmask(state), "(kỳ vọng", expected, ")")

'++++' -> True (kỳ vọng True )
'+' -> False (kỳ vọng False )


**Độ phức tạp Cách 3:** **Time `O(S · n)`** (mỗi trạng thái xét `n` vị trí với thao tác bit `O(1)`), **Space `O(S)`**.

### Cách 4 — Trạng thái chuẩn hóa: đa tập độ dài các đoạn `+`

**Giải thích.** Khóa là `tuple` **đã sắp xếp** các độ dài đoạn. Nước đi: chọn một đoạn dài `k >= 2` và một vị trí lật tách nó thành `a` và `k - 2 - a` (với `a` từ `0` đến `(k-2)//2` do đối xứng), thay đoạn đó bằng hai đoạn mới (bỏ đoạn rỗng/độ dài `0`; đoạn độ dài `1` cũng vô dụng nhưng giữ lại hoặc bỏ đều đúng — ta bỏ các đoạn `< 2` vì chúng không có nước đi).
#### Code demo

In [11]:
def can_win_segments(state):
    @lru_cache(maxsize=None)
    def win(segs):                                          # segs: tuple độ dài các đoạn (>= 2), đã sắp xếp
        for idx, k in enumerate(segs):
            if idx > 0 and segs[idx - 1] == k:
                continue                                    # đoạn trùng độ dài cho cùng kết quả: bỏ qua
            rest = segs[:idx] + segs[idx + 1:]
            for a in range(0, (k - 2) // 2 + 1):            # tách k thành a và k - 2 - a (đối xứng nên a <= b)
                b = k - 2 - a
                new = list(rest) + [x for x in (a, b) if x >= 2]
                if not win(tuple(sorted(new))):
                    return True
        return False

    initial = tuple(sorted(L for L in segments(state) if L >= 2))
    result = win(initial)
    win.cache_clear()
    return result


for state, expected in examples:
    print(repr(state), "->", can_win_segments(state), "(kỳ vọng", expected, ")")

'++++' -> True (kỳ vọng True )
'+' -> False (kỳ vọng False )


**Độ phức tạp Cách 4:** số trạng thái = số **phân hoạch** (đa tập độ dài) có tổng `<=` tổng độ dài các đoạn ban đầu — **nhỏ hơn rất nhiều** số trạng thái dạng chuỗi/mặt nạ; **Space `O(P)`** với `P` là số đa tập có thể có. Dễ chạy trên `n = 60`.

### Cách 5 — Sprague–Grundy (tính số Grundy từng đoạn)

**Giải thích.** `g[0] = g[1] = 0`; với `k >= 2`: `g[k] = mex{ g[a] ^ g[k-2-a] : a = 0..k-2 }`. Người đi trước thắng ⟺ XOR của `g[len]` trên mọi đoạn `+` **khác `0`**.
#### Code demo

In [12]:
def grundy_table(limit):
    g = [0] * (limit + 1)
    for k in range(2, limit + 1):
        g[k] = mex({g[a] ^ g[k - 2 - a] for a in range(0, k - 1)})
    return g


G = grundy_table(60)
print(G[:21])           # số Grundy của các đoạn dài 0..20


def can_win_grundy(state):
    total = 0
    for length in segments(state):
        total ^= G[length]
    return total != 0


for state, expected in examples:
    print(repr(state), "->", can_win_grundy(state), "(kỳ vọng", expected, ")")

[0, 0, 1, 1, 2, 0, 3, 1, 1, 0, 3, 3, 2, 2, 4, 0, 5, 2, 2, 3, 3]
'++++' -> True (kỳ vọng True )
'+' -> False (kỳ vọng False )


**Độ phức tạp Cách 5:** dựng bảng `g` tới `n`: **`O(n²)`**; mỗi truy vấn **`O(n)`**; **Space `O(n)`**. Đây là cách **không phụ thuộc kích thước cây trò chơi**.

> Cách 3 là bản nộp phổ biến (mặt nạ bit). Cách 4 và 5 cho thấy tận dụng cấu trúc "tổng các trò chơi độc lập". Cách 1 chỉ để hiểu luật.

## 8. TRACE từng bước

Ta (a) vẽ cây minimax của `"++++"` (kèm kết luận thắng/thua mỗi nút) và (b) lập bảng số Grundy cho các đoạn dài `0..8` kèm tập các giá trị dùng trong `mex`.
#### Code demo

In [13]:
def trace_minimax(state, depth=0, player="người đi trước"):
    pad = "  " * depth
    nxts = moves(state)
    if not nxts:
        print(f"{pad}{state!r}: {player} KHÔNG đi được -> THUA")
        return False
    result = False
    print(f"{pad}{state!r}: {player} có {len(nxts)} nước đi")
    for nxt in nxts:
        other = "người đi sau" if player == "người đi trước" else "người đi trước"
        child = trace_minimax(nxt, depth + 1, other)
        print(f"{pad}  -> sau nước đi tới {nxt!r}: đối thủ {'THẮNG' if child else 'THUA'}")
        if not child:
            result = True
    print(f"{pad}=> {state!r}: {player} {'THẮNG' if result else 'THUA'}")
    return result


trace_minimax("++++")

'++++': người đi trước có 3 nước đi
  '--++': người đi sau có 1 nước đi
    '----': người đi trước KHÔNG đi được -> THUA
    -> sau nước đi tới '----': đối thủ THUA
  => '--++': người đi sau THẮNG
  -> sau nước đi tới '--++': đối thủ THẮNG
  '+--+': người đi sau KHÔNG đi được -> THUA
  -> sau nước đi tới '+--+': đối thủ THUA
  '++--': người đi sau có 1 nước đi
    '----': người đi trước KHÔNG đi được -> THUA
    -> sau nước đi tới '----': đối thủ THUA
  => '++--': người đi sau THẮNG
  -> sau nước đi tới '++--': đối thủ THẮNG
=> '++++': người đi trước THẮNG


True

In [14]:
def trace_grundy(limit):
    g = [0] * (limit + 1)
    print(" k | tập {g[a]^g[k-2-a]} | mex = g[k]")
    for k in range(limit + 1):
        if k >= 2:
            values = sorted({g[a] ^ g[k - 2 - a] for a in range(0, k - 1)})
            g[k] = mex(values)
        else:
            values = []
        print(f"{k:>2} | {str(values):<20} | {g[k]}")
    return g


trace_grundy(10)

 k | tập {g[a]^g[k-2-a]} | mex = g[k]
 0 | []                   | 0
 1 | []                   | 0
 2 | [0]                  | 1
 3 | [0]                  | 1
 4 | [0, 1]               | 2
 5 | [1]                  | 0
 6 | [0, 1, 2]            | 3
 7 | [0, 2]               | 1
 8 | [0, 3]               | 1
 9 | [1, 3]               | 0
10 | [0, 1, 2]            | 3


[0, 0, 1, 1, 2, 0, 3, 1, 1, 0, 3]

Bảng tóm tắt cây minimax (`"++++"`):

| trạng thái sau nước đi | đối thủ còn nước đi? | kết quả cho đối thủ |
|---|---|---|
| `"--++"` | có (một cặp) | thắng (đối thủ lật nốt, ta hết nước) |
| `"+--+"` | **không** | **thua** |
| `"++--"` | có (một cặp) | thắng |

Có một nước đi đẩy đối thủ vào thế thua ⇒ `"++++"` là thế **thắng**. Với Grundy: đoạn dài `4` có `g[4] = 2 ≠ 0` ⇒ thắng (khớp).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (minimax có nhớ, trạng thái mặt nạ bit). Đúng `class Solution`, đúng `canWin(self, currentState: str) -> bool`.

In [15]:
class Solution:
    def canWin(self, currentState: str) -> bool:
        @lru_cache(None)
        def dfs(mask: int) -> bool:
            for i in range(n - 1):
                if (mask & (1 << i)) == 0 or (mask & (1 << (i + 1))) == 0:     # cần hai '+' liền kề
                    continue
                if dfs(mask ^ (1 << i) ^ (1 << (i + 1))):                      # đối thủ thắng sau nước này: thử nước khác
                    continue
                return True                                                    # tìm được nước đẩy đối thủ vào thế thua
            return False

        mask, n = 0, len(currentState)
        for i, c in enumerate(currentState):
            if c == "+":
                mask |= 1 << i
        return dfs(mask)


sol = Solution()
for state, expected in examples:
    assert sol.canWin(state) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Minimax:** trò chơi hữu hạn (mỗi nước đi giảm số `'+'` đi `2`) không thể hòa; mọi trạng thái hoặc thắng hoặc thua cho người sắp đi. Quy nạp theo số `'+'` còn lại: nếu không có nước đi thì người đó thua (đúng luật); nếu có, trạng thái là thắng ⟺ tồn tại nước đi tới trạng thái thua của đối thủ (đối thủ đi trạng thái đó như người sắp đi).
- **Ghi nhớ:** kết quả của một trạng thái chỉ phụ thuộc chính trạng thái (không phụ thuộc đường đi tới nó), nên lưu lại và dùng lại là hợp lệ. Mặt nạ bit biểu diễn **một-một** các chuỗi.
- **Chuẩn hóa:** hai trạng thái có cùng đa tập độ dài các đoạn `'+'` (khác vị trí) có cây trò chơi **đẳng cấu**, nên cùng kết quả; đoạn độ dài `< 2` không có nước đi nên bỏ đi không ảnh hưởng.
- **Sprague–Grundy:** trò chơi là **tổng** của các đoạn độc lập (mỗi nước đi chỉ động vào một đoạn). Định lý: số Grundy của tổng = XOR các số Grundy; vị trí **thua** ⟺ số Grundy `= 0`. Số Grundy của đoạn dài `k` là `mex` của số Grundy các trạng thái sau một nước (hai đoạn `a`, `k - 2 - a` ⇒ `g[a] ^ g[k-2-a]`).
- **Dừng sớm:** khi gặp nước đi thắng ta trả `True` ngay: không cần xét nước đi khác.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(currentState)` (tối đa `60`), `S` số trạng thái khác nhau đạt được (chuỗi/mặt nạ), `P` số đa tập độ dài đoạn.

| Cách | Time | Space |
|---|---|---|
| 1. Minimax thuần | `O(n!!)` (bùng nổ) | `O(n²)` |
| 2. Có nhớ (chuỗi) | `O(S·n²)` | `O(S·n)` |
| 3. Có nhớ (mặt nạ bit) | `O(S·n)` | `O(S)` |
| 4. Trạng thái chuẩn hóa | `O(P·n²)` (`P ≪ S`) | `O(P)` |
| 5. Sprague–Grundy | `O(n²)` dựng bảng + `O(n)` mỗi truy vấn | `O(n)` |

**Trả lời follow-up:** với một đoạn `n` dấu `+`, số trạng thái dạng mặt nạ tăng như dãy Fibonacci (`~ φⁿ`) nên Cách 3 là **mũ** `O(φⁿ · n)` trong worst case dù có nhớ; nhờ ràng buộc "tối đa `20` dấu `+` liên tiếp" và việc dừng sớm nên đủ nhanh trên LeetCode. Cách 5 đa thức là tốt nhất về lý thuyết.

## 12. Bộ test

`oracle` là brute force độc lập: **minimax thuần không nhớ**, viết lại theo kiểu "đếm nước đi thua của đối thủ" (khác Cách 1 ở cấu trúc), chỉ dùng cho chuỗi có tối đa ~14 dấu `+`. Với chuỗi dài hơn, đối chiếu Cách 3, 4, 5 với nhau (độc lập về thuật toán: bit-mask vs đa tập vs Grundy).
#### Code demo

In [16]:
def oracle(state):
    """Minimax thuần: đếm số nước đi dẫn tới thế THUA của đối thủ; thắng nếu số đó > 0."""
    winning_moves = 0
    for i in range(len(state) - 1):
        if state[i : i + 2] == "++":
            after = state[:i] + "--" + state[i + 2 :]
            if oracle(after) is False:
                winning_moves += 1
    return winning_moves > 0


impls = {
    "minimax": can_win_minimax,
    "memo": can_win_memo,
    "bitmask": can_win_bitmask,
    "segments": can_win_segments,
    "grundy": can_win_grundy,
    "submit": sol.canWin,
}

for state, expected in examples:
    assert oracle(state) == expected
    for key, fn in impls.items():
        assert fn(state) == expected, (key, state, fn(state), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [17]:
edge_cases = [
    ("+", False, "một dấu +"),
    ("-", False, "một dấu -"),
    ("--------", False, "toàn dấu - (không có nước đi)"),
    ("+-+-+", False, "các + bị chặn, không có cặp ++"),
    ("++", True, "đúng một cặp: đi một nước là thắng"),
    ("+++", True, "hai nước đi, đều thắng"),
    ("++++", True, "ví dụ 1: đi vào giữa"),
    ("+++++", False, "5 dấu +: mọi nước đi đều để lại thế thắng cho đối thủ"),
    ("++-++", False, "hai cặp rời nhau: người đi trước lật một cặp, đối thủ lật cặp còn lại"),
    ("++-+++", False, "đoạn 2 và đoạn 3: g[2] ^ g[3] = 1 ^ 1 = 0 nên người đi trước thua"),
    ("-++-", True, "một cặp bị bao bởi -"),
    ("++++++", True, "6 dấu +"),
]
for state, expected, label in edge_cases:
    assert oracle(state) == expected, (label, oracle(state), expected)
    for key, fn in impls.items():
        assert fn(state) == expected, (key, label, state, fn(state), expected)
    print("PASS", label)

PASS một dấu +
PASS một dấu -
PASS toàn dấu - (không có nước đi)
PASS các + bị chặn, không có cặp ++
PASS đúng một cặp: đi một nước là thắng
PASS hai nước đi, đều thắng
PASS ví dụ 1: đi vào giữa
PASS 5 dấu +: mọi nước đi đều để lại thế thắng cho đối thủ
PASS hai cặp rời nhau: người đi trước lật một cặp, đối thủ lật cặp còn lại
PASS đoạn 2 và đoạn 3: g[2] ^ g[3] = 1 ^ 1 = 0 nên người đi trước thua
PASS một cặp bị bao bởi -
PASS 6 dấu +


### 12.2. Tính chất và phản ví dụ

- Đoạn `1` hoặc rỗng không có nước đi ⇒ **thua**.
- **Đối xứng/đảo chuỗi:** đảo ngược chuỗi không đổi kết quả; thêm bớt dấu `-` ở hai đầu không đổi kết quả.
- Hoán vị các đoạn (giữ độ dài) không đổi kết quả (do tính độc lập).
- **Phản ví dụ** cho "đếm số cặp `++` rồi xét chẵn lẻ": với `"+++"` có `2` cặp (chẵn) nhưng người đi trước **thắng**.
#### Code demo

In [18]:
rng = random.Random(294)
for _ in range(300):
    n = rng.randint(1, 14)
    s = "".join(rng.choice("+-+") for _ in range(n))
    r = sol.canWin(s)
    assert sol.canWin(s[::-1]) == r
    assert sol.canWin("-" + s + "-") == r
    parts = [p for p in s.split("-") if p]
    rng.shuffle(parts)
    assert sol.canWin("-".join(parts)) == r
print("PASS tính chất (đảo chuỗi, thêm -, hoán vị các đoạn) trên 300 chuỗi ngẫu nhiên")


def sai_chan_le(state):
    pairs = sum(1 for i in range(len(state) - 1) if state[i] == state[i + 1] == "+")
    return pairs % 2 == 1                                           # SAI: nước đi thay đổi số cặp theo cách khó đoán


bad = "+++"
print("phản ví dụ", repr(bad), ": đúng", sol.canWin(bad), "| theo chẵn lẻ số cặp:", sai_chan_le(bad))
assert sai_chan_le(bad) != sol.canWin(bad)
wrong_cases = [s for s in ("++", "+++", "++++", "+++++", "++++++", "++-++") if sai_chan_le(s) != sol.canWin(s)]
print("các chuỗi mà quy tắc chẵn lẻ sai:", wrong_cases)

PASS tính chất (đảo chuỗi, thêm -, hoán vị các đoạn) trên 300 chuỗi ngẫu nhiên
phản ví dụ '+++' : đúng True | theo chẵn lẻ số cặp: False
các chuỗi mà quy tắc chẵn lẻ sai: ['+++']


### 12.3. Đối chiếu ngẫu nhiên (có seed) và chuỗi dài
#### Code demo

In [19]:
rng = random.Random(4294)
for trial in range(1500):
    n = rng.randint(1, 14)
    s = "".join(rng.choice("++-") for _ in range(n))
    truth = oracle(s)
    for key, fn in impls.items():
        assert fn(s) == truth, (key, s, fn(s), truth)
print("PASS 1500 chuỗi ngẫu nhiên dài tới 14: mọi cách khớp oracle")

# Số Grundy theo Cách 5 khớp minimax cho một đoạn dài k (k <= 13), chứng tỏ bảng Grundy đúng
for k in range(0, 14):
    assert (G[k] != 0) == oracle("+" * k), k
print("PASS bảng Grundy: G[k] != 0 khớp minimax cho một đoạn dài k = 0..13")

# Chuỗi dài hơn (tới 60 ký tự, đoạn <= 20): so Cách 4 (đa tập) và Cách 5 (Grundy), và Cách 3 khi đủ nhanh
long_cases = ["+" * 20, "+" * 20 + "-" + "+" * 20, "+" * 20 + "-" + "+" * 20 + "-" + "+" * 17, "++-+++-++++-+++++-++++++-+++++++"]
for _ in range(40):
    parts = [rng.randint(1, 20) for _ in range(rng.randint(1, 5))]
    s = "-".join("+" * p for p in parts)
    if len(s) <= 60:
        long_cases.append(s)
for s in long_cases:
    r5 = can_win_grundy(s)
    r4 = can_win_segments(s)
    assert r4 == r5, (s, r4, r5)
print(f"PASS {len(long_cases)} chuỗi dài (tới 60 ký tự): Cách 4 (đa tập) khớp Cách 5 (Grundy)")

# Cách 3 (mặt nạ bit) trên vài chuỗi vừa phải, đo thời gian
for s in ["+" * 20, "+" * 24, "++-+++-++++-+++++-++++++"]:
    t0 = time.perf_counter()
    r3 = can_win_bitmask(s)
    dt = (time.perf_counter() - t0) * 1000
    assert r3 == can_win_grundy(s)
    print(f"mặt nạ bit trên {len(s)} ký tự ({s[:12]}...): {r3}, {dt:8.1f} ms")

PASS 1500 chuỗi ngẫu nhiên dài tới 14: mọi cách khớp oracle
PASS bảng Grundy: G[k] != 0 khớp minimax cho một đoạn dài k = 0..13


PASS 39 chuỗi dài (tới 60 ký tự): Cách 4 (đa tập) khớp Cách 5 (Grundy)
mặt nạ bit trên 20 ký tự (++++++++++++...): True,      4.9 ms
mặt nạ bit trên 24 ký tự (++++++++++++...): True,     33.9 ms
mặt nạ bit trên 24 ký tự (++-+++-++++-...): True,      2.0 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên điều kiện "không còn nước đi thì thua"
#### Code sai

In [20]:
def sai_khong_co_nuoc_di(state):
    for i in range(len(state) - 1):
        if state[i] == "+" and state[i + 1] == "+":
            if not sai_khong_co_nuoc_di(state[:i] + "--" + state[i + 2:]):
                return True
    return True                                          # SAI: hết nước đi / mọi nước đều thua phải trả False


print("'+' đúng:", sol.canWin("+"), "| sai:", sai_khong_co_nuoc_di("+"))
print("'+++++' đúng:", sol.canWin("+++++"), "| sai:", sai_khong_co_nuoc_di("+++++"))

'+' đúng: False | sai: True
'+++++' đúng: False | sai: True


**Vì sao sai:** nếu không tìm được nước đi nào đẩy đối thủ vào thế thua thì người chơi **thua** (kể cả khi không có nước đi). Phải `return False` ở cuối.

### Lỗi 2: Nhầm vai (không đảo kết quả của đối thủ)
#### Code sai

In [21]:
def sai_nham_vai(state):
    for i in range(len(state) - 1):
        if state[i] == "+" and state[i + 1] == "+":
            if sai_nham_vai(state[:i] + "--" + state[i + 2:]):          # SAI: thiếu 'not'
                return True
    return False


for s in ("++", "+++", "++++"):
    print(repr(s), "đúng:", sol.canWin(s), "| sai:", sai_nham_vai(s))

'++' đúng: True | sai: False
'+++' đúng: True | sai: False
'++++' đúng: True | sai: False


**Vì sao sai:** nếu đối thủ **thắng** ở trạng thái sau nước đi của ta thì đó là nước đi **xấu** cho ta. Ta cần nước đi sao cho đối thủ **thua**: `not dfs(next)`.

### Lỗi 3: Không ghi nhớ (hết thời gian)
#### Code sai

In [22]:
def dem_so_loi_goi(state):
    calls = 0

    def go(s):
        nonlocal calls
        calls += 1
        for i in range(len(s) - 1):
            if s[i] == "+" and s[i + 1] == "+":
                if not go(s[:i] + "--" + s[i + 2:]):
                    return True
        return False

    go(state)
    return calls


for n in (5, 9, 10, 15):
    s = "+" * n
    print(f"{n} dấu + liền (thế {'thua' if not sol.canWin(s) else 'thắng'}): số lời gọi không nhớ = {dem_so_loi_goi(s)}")

5 dấu + liền (thế thua): số lời gọi không nhớ = 9
9 dấu + liền (thế thua): số lời gọi không nhớ = 87
10 dấu + liền (thế thắng): số lời gọi không nhớ = 114
15 dấu + liền (thế thua): số lời gọi không nhớ = 5969


**Vì sao sai (chậm):** không nhớ nên cùng một trạng thái được giải lại nhiều lần; số lời gọi tăng rất nhanh theo `n` (với thế **thua** như `n = 9, 15` phải duyệt hết cây vì không có nước thắng để dừng sớm; ở thế thắng chương trình dừng sớm nên ít lời gọi hơn). **Code đúng:** `lru_cache` hoặc từ điển `memo`.

### Lỗi 4: Lật chỉ một ký tự (không đúng "++" → "--")
#### Code sai

In [23]:
def sai_lat_mot_ky_tu(state):
    for i in range(len(state) - 1):
        if state[i] == "+" and state[i + 1] == "+":
            nxt = state[:i] + "-+" + state[i + 2:]               # SAI: chỉ lật dấu đầu, dấu thứ hai vẫn là '+'
            if not sai_lat_mot_ky_tu(nxt):
                return True
    return False


for s in ("++++", "+++++"):
    print(repr(s), "đúng:", sol.canWin(s), "| sai:", sai_lat_mot_ky_tu(s))

'++++' đúng: True | sai: False
'+++++' đúng: False | sai: True


**Vì sao sai:** đề lật **cả hai** ký tự `"++"` thành `"--"`. Nếu chỉ lật một ký tự thì trò chơi khác hẳn (nước đi ít "tốn" hơn, các đoạn không bị tách đúng cách) và kết quả sai. Cũng cẩn thận giữ **độ dài chuỗi** không đổi.

### Lỗi 5: Kết luận theo tính chẵn lẻ số cặp `"++"`
Xem `sai_chan_le` ở phần test: `"+++"` có `2` cặp nhưng người đi trước vẫn thắng (một nước đi **phá** cả hai cặp). Mỗi nước đi có thể xoá **1, 2 hoặc 3** cặp `"++"` (cặp đã lật và các cặp chồng lên nó), nên không có quy luật chẵn lẻ đơn giản — cần minimax hoặc Sprague–Grundy.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**hai người chơi luân phiên, ai không đi được thì thua, chơi tối ưu**" → **minimax**: `thắng ⟺ tồn tại nước đi tới thế thua của đối thủ`.
- Trạng thái nhỏ gọn (≤ ~30 bit) → **mặt nạ bit + `lru_cache`**.
- Trò chơi chia được thành **các phần độc lập** (đoạn `+` bị `-` chặn) → **Sprague–Grundy**: `g = mex(g[a] ^ g[b])`, thắng ⟺ XOR `≠ 0`.
- **Dừng sớm** khi tìm được nước thắng.
- Họ hàng: **Flip Game** (293, liệt kê trạng thái sau một nước), **Nim Game** (292), **Stone Game** (877), **Can I Win** (464, mặt nạ bit), **Predict the Winner** (486).
- Luôn thử `"+"`, `"++"`, `"+++"`, `"++++"`, `"+++++"` bằng tay để kiểm tra trực giác.

## 15. Biến thể của bài

1. **Flip Game** (LeetCode 293): liệt kê mọi trạng thái sau **một** nước đi (`moves`).
2. **Can I Win** (464): trò chơi chọn số, mặt nạ bit + memo.
3. **Nim** và các biến thể: XOR của các đống.
4. **Kayles / Dawson's Kayles**: trò chơi cùng họ (bài này chính là Dawson's Kayles với luật tách).
5. **In ra một nước đi thắng** (không chỉ `True/False`).
6. **Số lượng nước đi thắng** từ trạng thái cho trước.
#### Code demo

In [24]:
def winning_move(state):
    """Trả về trạng thái sau một nước đi thắng (hoặc None)."""
    for nxt in moves(state):
        if not can_win_grundy(nxt):
            return nxt
    return None


def count_winning_moves(state):
    return sum(1 for nxt in moves(state) if not can_win_grundy(nxt))


print(winning_move("++++"), count_winning_moves("++++"), winning_move("+++++"))        # '+--+' 1 None
print(moves("++++"))

+--+ 1 None
['--++', '+--+', '++--']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax thuần | thử mọi nước đi đệ quy | mũ/giai thừa | `O(n²)` | hiểu luật, oracle |
| 2. Có nhớ (chuỗi) | `memo[state]` | `O(S·n²)` | `O(S·n)` | dễ viết |
| 3. Có nhớ (mặt nạ bit) | state là số nguyên | `O(S·n)` | `O(S)` | **bản nộp phổ biến** |
| 4. Đa tập độ dài đoạn | chuẩn hóa, bỏ vị trí | `O(P·n²)` | `O(P)` | giảm mạnh số trạng thái |
| 5. Sprague–Grundy | `g[k] = mex(g[a]^g[b])`, XOR | `O(n²)` | `O(n)` | **đa thức**, tối ưu về lý thuyết |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `moves("+++")`; (b) `can_win_demo("++")`; (c) `mex([0, 1, 2, 4])`; (d) `segments("++-+-+++")`.

**Bài 2 (tính bằng tay).** Xác định người đi trước thắng hay thua với `"+++++"` bằng cách xét mọi nước đi (gợi ý: các trạng thái sau một nước là `"--+++"`, `"+--++"`, `"++--+"`, `"+++--"`).

**Bài 3 (điền TODO).** Hoàn thiện `win_todo(state)` bằng minimax có nhớ.
#### Code demo

In [25]:
def win_todo(state):
    memo = {}

    def win(s):
        # TODO 1: nếu s đã có trong memo thì trả memo[s]
        # TODO 2: với mỗi i có s[i] == s[i + 1] == "+": nxt = s[:i] + "--" + s[i + 2:]
        # TODO 3: nếu not win(nxt): memo[s] = True và trả True
        # TODO 4: memo[s] = False và trả False
        pass

    # TODO 5: trả win(state)
    pass


# Tự làm xong rồi mới mở test:
# for state, expected in examples:
#     assert win_todo(state) == expected

**Bài 4 (tự viết lại).** Viết `grundy_of_state(state)` trả về **số Grundy tổng** (XOR) của một chuỗi và `losing_lengths(limit)` liệt kê các độ dài `k` của một đoạn mà người đi trước **thua** (`g[k] = 0`).

**Bài 5 (biến thể / mini challenge).** Viết `flip_game_i(state)` (LeetCode 293) trả mọi trạng thái sau **một** nước đi, và `best_first_move(state)` trả **một** nước đi thắng bằng Sprague–Grundy (chọn nước đi làm XOR mới bằng `0`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `moves` lật từng cặp `++`; `mex` là số nhỏ nhất vắng mặt. Bài 2: một trạng thái là thua nếu **mọi** nước đi dẫn tới trạng thái thắng cho đối thủ. Bài 4: `losing_lengths` kiểm tra `g[k] == 0`.
- **Mức 2 (vừa):** Bài 1: `['--+', '+--']`, `True`, `3`, `[1, 2, 3]` (chuỗi `"++-+-+++"` có các đoạn `+` độ dài `2`, `1`, `3`). Bài 2: kiểm tra từng trạng thái sau nước đi bằng `can_win_grundy`.
- **Mức 3 (sâu):** Bài 2: `"+++++"`: sau nước đi ta có hai đoạn tổng `3`: `(0,3)`, `(1,2)`, `(2,1)`, `(3,0)`; số Grundy `g[3] = 1`, `g[1]^g[2] = 1`, đều `≠ 0` ⇒ đối thủ thắng ⇒ `"+++++"` thua. Bài 5: nước đi thắng làm tổng Grundy `= 0`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert moves("+++") == ["--+", "+--"] and can_win_demo("++") is True
assert mex([0, 1, 2, 4]) == 3 and segments("++-+-+++") == [1, 2, 3]

# Bài 2
assert all(can_win_grundy(nxt) for nxt in moves("+++++")) and sol.canWin("+++++") is False


# Bài 3
def win_todo_ans(state):
    memo = {}

    def win(s):
        if s in memo:
            return memo[s]
        for i in range(len(s) - 1):
            if s[i] == s[i + 1] == "+":
                if not win(s[:i] + "--" + s[i + 2:]):
                    memo[s] = True
                    return True
        memo[s] = False
        return False

    return win(state)


for state, expected in examples:
    assert win_todo_ans(state) == expected


# Bài 4
def grundy_of_state(state):
    total = 0
    for length in segments(state):
        total ^= G[length]
    return total


def losing_lengths(limit):
    return [k for k in range(limit + 1) if G[k] == 0]


assert grundy_of_state("++++") == G[4] and grundy_of_state("++-++") == 0
print("độ dài đoạn mà người đi trước thua (k <= 20):", losing_lengths(20))


# Bài 5
def flip_game_i(state):
    return moves(state)


def best_first_move(state):
    for nxt in flip_game_i(state):
        if grundy_of_state(nxt) == 0:
            return nxt
    return None


assert best_first_move("++++") == "+--+"
assert best_first_move("+++++") is None
for _ in range(200):
    n = random.randint(1, 20)
    s = "".join(random.choice("+-+") for _ in range(n))
    mv = best_first_move(s)
    assert (mv is not None) == can_win_grundy(s)
print("PASS đáp án bài tập")

độ dài đoạn mà người đi trước thua (k <= 20): [0, 1, 5, 9, 15]
PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (minimax + mặt nạ bit + `lru_cache`)

```python
class Solution:
    def canWin(self, currentState: str) -> bool:
        @lru_cache(None)
        def dfs(mask):
            for i in range(n - 1):
                if mask >> i & 1 and mask >> (i + 1) & 1:
                    if not dfs(mask ^ (1 << i) ^ (1 << (i + 1))):
                        return True
            return False

        n = len(currentState)
        mask = sum(1 << i for i, c in enumerate(currentState) if c == "+")
        return dfs(mask)
```

**Checklist**

- [ ] Hiểu: thắng ⟺ **tồn tại** nước đi tới thế **thua** của đối thủ (`not dfs(next)`)?
- [ ] Hết nước đi ⇒ **thua** (`return False` ở cuối)?
- [ ] Lật **hai** ký tự thành `"--"` (giữ độ dài)?
- [ ] Có **ghi nhớ** trạng thái (`lru_cache` / `memo`)?
- [ ] Biết cách tối ưu hơn: chuẩn hóa đa tập đoạn hoặc **Sprague–Grundy** (XOR số Grundy)?
- [ ] Đã thử `"+"`, `"++"`, `"+++"`, `"++++"`, `"+++++"`, `"++-++"`?
- [ ] Đã đối chiếu với minimax thuần trên chuỗi ngắn ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def canWin(self, currentState: str) -> bool`?